# Victoria electricity with temperature covariates

This GLM-style example forecasts a held-out week of hourly electricity demand. Known future temperature enters alongside daily and weekly Fourier features, while a Student-T likelihood makes the fit robust to demand spikes.

In [ ]:
import logging

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

from pymc_forecast import Forecaster, Horizon, evaluate_forecast, predict
from pymc_forecast.data import FUTURE_DIM, TIME_DIM
from pymc_forecast.datasets import load_victoria_electricity

az.style.use("arviz-darkgrid")
logging.getLogger("pymc").setLevel(logging.ERROR)
SEED = 42

## Demand and known future features

The bundled data contains eight weeks at hourly frequency. The covariate array retains the real datetime coordinate across both the training prefix and forecast week.

In [ ]:
demand_raw, temperature = load_victoria_electricity()
demand = np.log(demand_raw).rename("log_demand")
n = demand.sizes[TIME_DIM]
position = np.arange(n)
temperature_z = (temperature - temperature.mean()) / temperature.std()


def fourier(period: float, terms: int) -> tuple[np.ndarray, list[str]]:
    harmonic = np.arange(1, terms + 1)
    angle = 2 * np.pi * position[:, None] * harmonic[None, :] / period
    values = np.concatenate([np.sin(angle), np.cos(angle)], axis=1)
    names = [f"sin_{period:g}_{k}" for k in harmonic] + [f"cos_{period:g}_{k}" for k in harmonic]
    return values, names


daily, daily_names = fourier(24, 3)
weekly, weekly_names = fourier(24 * 7, 3)
feature_values = np.column_stack([temperature_z, temperature_z**2, daily, weekly])
feature_names = ["temperature", "temperature_squared", *daily_names, *weekly_names]
covariates = xr.DataArray(
    feature_values,
    dims=(TIME_DIM, "covariate"),
    coords={TIME_DIM: demand[TIME_DIM], "covariate": feature_names},
)

test_window = 24 * 7
split = n - test_window
demand_train = demand.isel({TIME_DIM: slice(None, split)})
demand_test = demand.isel({TIME_DIM: slice(split, None)}).rename({TIME_DIM: FUTURE_DIM})
covariates_train = covariates.isel({TIME_DIM: slice(None, split)})
print(covariates.sizes)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(demand_raw[TIME_DIM], demand_raw, color="C0")
axes[0].set(ylabel="demand (GW)")
axes[1].plot(temperature[TIME_DIM], temperature, color="C1")
axes[1].set(ylabel="temperature (°C)", xlabel="time")
for ax in axes:
    ax.axvline(demand[TIME_DIM].values[split], color="black", ls="--")
fig.suptitle("Victoria electricity: train/test split")
plt.show()

## Robust regression model

The regression weights carry the named `covariate` dimension. `predict` slices the same linear predictor into `time` and `time_future` likelihood variables.

In [ ]:
center = float(demand_train.mean())


def electricity_model(h: Horizon, covariates: xr.DataArray) -> None:
    intercept = pm.Normal("intercept", center, 0.5)
    weight_scale = pm.HalfNormal("weight_scale", 0.5)
    weight = pm.Normal("weight", 0.0, weight_scale, dims="covariate")
    sigma = pm.HalfNormal("sigma", 0.2)
    nu = pm.Gamma("nu", alpha=10, beta=2)
    mu = intercept + pt.dot(covariates.values, weight)
    predict(
        h,
        lambda name, eta, dims, observed: pm.StudentT(
            name, nu=nu, mu=eta, sigma=sigma, dims=dims, observed=observed
        ),
        mu,
    )

## Forecast with future temperatures

The full covariate array supplies the held-out week's observed temperatures as a scenario. The returned forecast uses the same datetime values on `time_future`.

In [ ]:
forecaster = Forecaster(
    electricity_model,
    demand_train,
    covariates_train,
    num_steps=8_000,
    random_seed=SEED,
)
idata = forecaster.forecast(covariates, num_samples=500, random_seed=SEED)
forecast = idata["predictions"]["forecast"]
print(evaluate_forecast(forecast, demand_test))
print(forecast.sizes)

In [ ]:
samples = np.exp(forecast).stack(sample=("chain", "draw"))
mean = samples.mean("sample")
lo, hi = samples.quantile([0.05, 0.95], dim="sample").values

fig, ax = plt.subplots(figsize=(12, 5))
history = demand_raw.isel({TIME_DIM: slice(split - 24 * 3, None)})
ax.plot(history[TIME_DIM], history, color="black", lw=1, label="observed")
ax.plot(forecast[FUTURE_DIM], mean, color="C1", label="forecast mean")
ax.fill_between(forecast[FUTURE_DIM], lo, hi, color="C1", alpha=0.25, label="90% interval")
ax.axvline(forecast[FUTURE_DIM].values[0], color="gray", ls="--")
ax.set(title="Victoria electricity forecast", xlabel="time", ylabel="demand (GW)")
ax.legend()
plt.show()